# Basic checks of CREDIT ERA5 inputs

In [1]:
import numpy as np
import xarray as xr
from glob import glob

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [15]:
fn_new = '/glade/derecho/scratch/ksha/CREDIT/RAW_OUTPUT/wxformer_6h_test/2021-05-01T00Z/pred_2021-05-01T00Z_006.nc'
fn_old = '/glade/derecho/scratch/ksha/CREDIT/RAW_OUTPUT/wxformer_6h/2021-05-01T00Z/pred_2021-05-01T00Z_006.nc'

In [16]:
ds_old = xr.open_dataset(fn_old)

In [17]:
ds_new = xr.open_dataset(fn_new)

In [18]:
ds_old['Z500'].values - ds_new['Z500'].values

array([[[-0.03515625, -0.05859375, -0.0703125 , ..., -0.0859375 ,
         -0.06640625,  0.03515625],
        [-0.03125   , -0.0625    , -0.07421875, ..., -0.14453125,
         -0.109375  , -0.03515625],
        [-0.09765625, -0.07421875, -0.0859375 , ..., -0.1328125 ,
         -0.09375   , -0.04296875],
        ...,
        [ 0.125     ,  0.2734375 ,  0.23828125, ...,  0.37890625,
          0.359375  ,  0.328125  ],
        [ 0.11328125,  0.2109375 ,  0.1875    , ...,  0.375     ,
          0.34765625,  0.3515625 ],
        [ 0.046875  ,  0.171875  ,  0.171875  , ...,  0.3359375 ,
          0.34765625,  0.3359375 ]]], dtype=float32)

## Variable name and file size consistency checks

In [3]:
# filenames = sorted(glob('/glade/derecho/scratch/ksha/CREDIT_data/ERA5_plevel_base/upper_air/*.zarr'))

# for i_fn, fn in enumerate(filenames):
#     try:
#         ds_temp = xr.open_zarr(fn)
#         # print(list(ds_temp.keys()))
#         variable_sizes = [var.size for var_name, var in ds_temp.data_vars.items()]
#         print(variable_sizes)
#     except:
#         print(fn)

In [4]:
# filenames = sorted(glob('/glade/derecho/scratch/ksha/CREDIT_data/ERA5_plevel_base/accum/*.zarr'))

# for i_fn, fn in enumerate(filenames):
#     try:
#         ds_temp = xr.open_zarr(fn)
#         # print(list(ds_temp.keys()))
#         variable_sizes = [var.size for var_name, var in ds_temp.data_vars.items()]
#         print(variable_sizes)
#     except:
#         print(fn)

In [5]:
# filenames = sorted(glob('/glade/derecho/scratch/ksha/CREDIT_data/ERA5_plevel_base/surf/*.zarr'))

# for i_fn, fn in enumerate(filenames):
#     try:
#         ds_temp = xr.open_zarr(fn)
#         # print(list(ds_temp.keys()))
#         variable_sizes = [var.size for var_name, var in ds_temp.data_vars.items()]
#         print(variable_sizes)
#     except:
#         print(fn)

## NaN checks

In [16]:
# def check_nans_ds(ds):
#     return ds.to_array().isnull().any().compute().item()
#     # return bool(ds.to_array().isnull().any().compute())

def check_nans_ds(ds):
    nan_vars = []
    for var in ds.data_vars:
        # Check if there are any NaNs in the variable
        if ds[var].isnull().any():
            nan_vars.append(var)
    return nan_vars

In [ ]:
# filenames = sorted(
#     glob('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/refcst/*.zarr'))

# for i_fn, fn in enumerate(filenames):
#     ds_temp = xr.open_zarr(fn)
#     nan_vars = check_nans_ds(ds_temp)
    
#     if nan_vars:
#         print('Dataset contains NaNs in the following variables:')
#         for var in nan_vars:
#             print(f"- {nan_vars}")
#         print(f"File: {fn}")
#     else:
#         print('Dataset does not contain NaNs')

In [5]:
# filenames = sorted(
#     glob('/glade/derecho/scratch/ksha/CREDIT_data/ERA5_mlevel_1deg/cloud/ERA5_mlevel_1deg_6h_cloud_*_conserve.zarr'))

# for i_fn, fn in enumerate(filenames):
#     ds_temp = xr.open_zarr(fn)
#     nan_vars = check_nans_ds(ds_temp)
    
#     if nan_vars:
#         print('Dataset contains NaNs in the following variables:')
#         for var in nan_vars:
#             print(f"- {nan_vars}")
#         print(f"File: {fn}")
#     else:
#         print('Dataset does not contain NaNs')

## Mass-conserved integral check

In [7]:
path_subset = '/glade/derecho/scratch/ksha/CREDIT_data/ERA5_plevel_base/upper_subset/'
path_original = '/glade/derecho/scratch/ksha/CREDIT_data/ERA5_plevel_base/upper_air/'
subset_level = np.array([1, 250, 450, 550, 650, 750, 850, 950, 1000])

In [8]:
ds_subset = xr.open_zarr(path_subset+'ERA5_subset_6h_upper_L8_1980.zarr')
ds_original = xr.open_zarr(path_original+'ERA5_plevel_6h_upper_air_1980.zarr')

In [9]:
for i in range(721):
    for j in range(1440):
        
        T_subset = np.array(ds_subset['Q'].isel(time=999, latitude=i, longitude=j))
        int_subset = np.sum(T_subset*np.diff(subset_level))
        
        T_original = np.array(ds_original['Q'].isel(time=999, latitude=i, longitude=j))
        int_original = np.trapz(T_original, np.array(ds_original['level']))

        int_diff = np.abs(int_subset - int_original)
        
        if int_diff > 1e-7 or np.isnan(int_diff): 
            print('diff: {}, ix: {}, iy: {}'.format(i, j))
        
print('... done ...')

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x14f9002d4110>>
Traceback (most recent call last):
  File "/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 

KeyboardInterrupt



In [3]:
ds_subset = xr.open_zarr(path_subset+'ERA5_subset_6h_Q_1984.zarr')
ds_original = xr.open_zarr(path_original+'ERA5_plevel_6h_Q_1984.zarr')

In [4]:
for i in range(721):
    for j in range(1440):

        T_subset = np.array(ds_subset['specific_total_water'].isel(time=999, latitude=i, longitude=j))
        int_subset = np.sum(T_subset*np.diff(subset_level))
        
        T_original = np.array(ds_original['specific_total_water'].isel(time=999, latitude=i, longitude=j))
        int_original = np.trapz(T_original, np.array(ds_original['level']))

        int_diff = np.abs(int_subset - int_original)
        
        if int_diff > 1e-7: 
            print('diff: {}, ix: {}, iy: {}'.format(i, j))

print('... done ...')


KeyboardInterrupt

